In [58]:
import torch.nn as nn
import torch
import pandas as pd
import copy
import random
import numpy as np

In [59]:
max_len = 50
embed_dim = 256

In [60]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
set_seed()

In [61]:
df = pd.read_csv('/Users/baonguyen/IU/thesis/data/clean_data/data_with_bertopic_column.csv')
df['review_date'] = pd.to_datetime(df['review_date'])
df['month'] = df['review_date'].dt.month


In [62]:
df = df.apply(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x), axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192544 entries, 0 to 192543
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Unnamed: 0      192544 non-null  int64         
 1   fit             192544 non-null  object        
 2   user_id         192544 non-null  int64         
 3   bust size       192544 non-null  object        
 4   item_id         192544 non-null  int64         
 5   weight          192544 non-null  object        
 6   rating          192544 non-null  float64       
 7   rented for      192544 non-null  object        
 8   review_text     192544 non-null  object        
 9   body type       192544 non-null  object        
 10  review_summary  192544 non-null  object        
 11  category        192544 non-null  object        
 12  height          192544 non-null  object        
 13  size            192544 non-null  int64         
 14  age             192544 non-null  flo

In [63]:
df['review_date']=pd.to_datetime(df['review_date'])
df_sorted = df.sort_values('review_date')
df_sorted.rename(columns={'rented for':'rented_for','body type':'body_type','bust size':'bust_size'},inplace=True)

In [64]:
df_sorted.columns

Index(['Unnamed: 0', 'fit', 'user_id', 'bust_size', 'item_id', 'weight',
       'rating', 'rented_for', 'review_text', 'body_type', 'review_summary',
       'category', 'height', 'size', 'age', 'review_date', 'Topic', 'month'],
      dtype='object')

In [65]:
side_feature =  ['rented_for','Topic']

In [66]:
# item to index and vice versa
unique_item_id = set(df_sorted['item_id'])
item_to_index = {item:idx +1 for idx , item in enumerate(unique_item_id)}
index_to_item = {idx+1:item for idx , item in enumerate(unique_item_id)}

for i in side_feature:
    exec(f'unique_{i} = set(df_sorted[i])')
    exec(f'{i}_to_index = {{i:idx+1 for idx,i in enumerate(unique_{i})}}')
    exec(f'index_to_{i} = {{idx+1:i for idx,i in enumerate(unique_{i})}}')

In [67]:
# Step 1: Group and aggregate
user_item_sequence = (
    df_sorted.groupby('user_id')[['item_id']+side_feature]
    .agg(list)
    .to_dict(orient='index')
)

# Step 2: Remove users with fewer than 2 item_ids
user_item_sequence = {
    user: val
    for user, val in user_item_sequence.items()
    if len(val['item_id']) >= 2
}


In [68]:
user_item_to_index_sequence = {}

for user, value in user_item_sequence.items():
    user_dict = {
        'item_id': [item_to_index[item] for item in value['item_id']]
    }
    for i in side_feature:
        # Dynamically get the correct mapping dict by name
        mapping_dict = globals()[f"{i}_to_index"]
        user_dict[i] = [mapping_dict[a] for a in value[i]]
    user_item_to_index_sequence[user] = user_dict


In [69]:


def mask_sequence(sequence: dict, mask_ratio: float):
    labels = {}
    mask_seq = {}
    for user, seq in sequence.items():
        mask_seq[user] = copy.deepcopy(seq)  # Deep copy so original is untouched
        labels[user] = [-100] * len(seq['item_id'])
        for i in range(len(mask_seq[user]['item_id'])):
            if random.random() < mask_ratio:
                labels[user][i] = mask_seq[user]['item_id'][i]  # Save original item id
                mask_seq[user]['item_id'][i] = 0       # Mask the item id
                for feature in side_feature:
                    mask_seq[user][feature][i] = 0
                
    return mask_seq, labels


In [70]:

mask_seq , labels = mask_sequence(user_item_to_index_sequence,mask_ratio=0.35)


In [71]:
def padding(mask_seq, labels, max_len=64, pad_item = 0, pad_label=-100):
    """
    Pads all user sequences in mask_seq and labels to max_len.
    
    """
    def pad(seq, max_len, pad_value):
        if len(seq) < max_len:
            return seq + [pad_value] * (max_len - len(seq))
        else:
            return seq[len(seq)-max_len:len(seq)]
    
    padded_mask_seq = {}
    padded_labels = {}

    for user in mask_seq:
        padded_mask_seq[user] = {
            **{'item_id': pad(mask_seq[user]['item_id'], max_len, pad_item)},
            **{i: pad(mask_seq[user][i], max_len, pad_item) for i in side_feature}
        }

        padded_labels[user] = pad(labels[user], max_len, pad_label)
    
    return padded_mask_seq, padded_labels


In [72]:
# padded_mask_seq,padded_labels = padding(mask_seq,labels,max_len=max_len)

In [73]:
# import numpy as np
# class SinusoidalPositionalEncoding(nn.Module):
#     def __init__(self, hidden_size, max_len=5000):
#         super(SinusoidalPositionalEncoding, self).__init__()
#         position = torch.arange(0, max_len).unsqueeze(1)
#         div_term = torch.exp(torch.arange(0, hidden_size, 2) * -(np.log(10000.0) / hidden_size))
#         pe = torch.zeros(max_len, hidden_size)
#         pe[:, 0::2] = torch.sin(position * div_term)
#         pe[:, 1::2] = torch.cos(position * div_term)
#         pe = pe.unsqueeze(0)
#         self.register_buffer('pe', pe)
#     def forward(self, x):
#         seq_len = x.size(1)
#         return self.pe[:, :seq_len, :]


In [74]:
# tensor_item_ids = torch.stack([
#     torch.tensor(user_seq['item_id']) for user_seq in padded_mask_seq.values()
# ])
# tensor_topic_ids = torch.stack([
#     torch.tensor(user_seq['Topic']) for user_seq in padded_mask_seq.values()
# ])
# tensor_labels = torch.stack([
#     torch.tensor(seq) for seq in padded_labels.values()
#     ])
# train_dataset = torch.utils.data.TensorDataset(
#     tensor_item_ids,
#     tensor_topic_ids,
#     tensor_labels
# )
# train_dataloader = torch.utils.data.DataLoader(train_dataset,batch_size=32,shuffle=True)

In [75]:
class differentiable_attn_mask(nn.Module):
    def __init__(self,):
        super(differentiable_attn_mask).__init__()
    pass

# nova bert architecture

In [76]:
class GatingFusor(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(h, 1))

    def forward(self, features):   
        gates = torch.sigmoid(features @ self.weight)         
        fused = torch.sum(gates * features, dim=2)            
        return fused

In [77]:
class NovabertEmbedding(nn.Module):
    def __init__(self,num_item,num_side_feature_ids:dict,embedding_dim,max_len=64):
        super(NovabertEmbedding,self).__init__()
        self.item_embedding = nn.Embedding(num_item,embedding_dim)
        self.side_embedding_  = nn.ModuleDict()
        for feat_name,num_feat in num_side_feature_ids.items():
            self.side_embedding_[feat_name]=nn.Embedding(num_feat,embedding_dim)
        self.position_encoding = nn.Embedding(max_len, embedding_dim)
    def forward(self, item_ids, side_feature_ids: dict):
        position_ids = torch.arange(item_ids.size(1), dtype=torch.long, device=item_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(item_ids)
        item_embed = self.item_embedding(item_ids)
        pos_embed = self.position_encoding(position_ids)
        item_embed = item_embed + pos_embed
        side_emb_list = []

        for feat_name in self.side_embedding_:
            
            feat_ids = side_feature_ids[feat_name]   # <-- Access by key, get tensor
            side_embed = self.side_embedding_[feat_name](feat_ids) + pos_embed
            side_emb_list.append(side_embed)
            
        return item_embed, side_emb_list


In [78]:
class NovabertCrossAttention(nn.Module):
    def __init__(self,embedding_dim,num_heads=8):
        super(NovabertCrossAttention,self).__init__()
        self.num_heads = num_heads
        self.head_dim = embedding_dim //num_heads
        self.value_proj = nn.Linear(embedding_dim,embedding_dim)
        self.query_proj = nn.Linear(embedding_dim,embedding_dim)
        self.key_proj = nn.Linear(embedding_dim,embedding_dim)
        self.fusor = GatingFusor(h=embedding_dim)


        self.output_proj = nn.Sequential(
              
        )
    def forward(self,item_embed,side_feature_embed:list,attn_mask=None,key_padding_mask=None):
        batch_size , sequence_len , embedding_dim = item_embed.size()
        def reshape(x:torch.tensor):
            return x.view(batch_size,sequence_len,self.num_heads,self.head_dim).transpose(1,2)
        # Batch,num_head,sequence_len,head_dim  (B,H,L,D)
        # print(*side_feature_embed)
        
        
        features = torch.stack([item_embed] + side_feature_embed, dim=2)

        fused_features = self.fusor(features)
        V = self.value_proj(item_embed)
        Q = self.query_proj(fused_features)
        K = self.key_proj(fused_features)
        Q = reshape(Q)
        K = reshape(K)
        V = reshape(V)
        scores = torch.matmul(Q,K.transpose(-2,-1)) / np.sqrt(self.head_dim) # B,H,L,L 
        
        if attn_mask is not None:
            scores += attn_mask.unsqueeze(0)  # Broadcast across batch ??? **********
            pass
        if key_padding_mask is not None:
            key_padding_mask = key_padding_mask.unsqueeze(1).unsqueeze(2) # B,1,1,L
            scores = scores.masked_fill(key_padding_mask,float('-inf'))
            # print(scores)

        attn_weights = torch.softmax(scores,dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        attn_output = torch.matmul(attn_weights,V) 

        # concat
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size,sequence_len,embedding_dim)
        return self.output_proj(attn_output)

            

In [79]:
class NovabertLayer(nn.Module):
    def __init__(self,embedding_dim,num_heads):
        super(NovabertLayer,self).__init__()
        self.cross_attn = NovabertCrossAttention(embedding_dim, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.GELU(),
            nn.Linear(embedding_dim * 4, embedding_dim)
        )
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.2)
    def forward(self, id_embed,side_feature_embed:list , attention_mask=None,key_padding_mask = None):
        guided = self.cross_attn(id_embed,side_feature_embed,attention_mask,key_padding_mask)
        x = self.norm1(id_embed + self.dropout(guided))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

In [80]:
class NovabertModel(nn.Module):
    def __init__(self, num_items,num_side_feature_ids:dict, embedding_dim, max_len=64, num_layers=4, num_heads=8):
        super(NovabertModel, self).__init__()
        self.embedding = NovabertEmbedding(num_item=num_items,
                                           num_side_feature_ids=num_side_feature_ids,
                                           embedding_dim=embedding_dim,
                                           max_len=max_len)
        self.nova_layer = nn.ModuleList([
            NovabertLayer(embedding_dim, num_heads) 
            for _ in range(num_layers)
        ])
        
        self.output_layer = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(embedding_dim, num_items)
        )

    def forward(self, item_ids, side_feature_ids: dict, attention_mask=None, key_padding_mask=None):
        x, y = self.embedding(item_ids, side_feature_ids)
        for layer in self.nova_layer:
            x = layer(x, y, attention_mask, key_padding_mask)
        return self.output_layer(x)



In [81]:
def hit_ratio(ground_truth:list,prediction:list,k:int):
    hits = 0
    total = len(ground_truth)
    for i, (gt_item, pred) in enumerate(zip(ground_truth, prediction)):
        
        if gt_item in pred[:k]:
            print(f"[Sample {i}] GT: {gt_item}, Pred top-{k}: {pred[:k]}")
            hits += 1
    return hits / total
# -------------------------
# Function to load the popularity data (counts.csv)
def load_popularity_data(filepath):
    df = pd.read_csv(filepath)
    item_popularity = dict(zip(df['item_id'], df['count']))  # Item popularity dictionary
    total_count = sum(item_popularity.values())  # Total count of interactions
    item_probabilities = {item: count / total_count for item, count in item_popularity.items()}  # Normalize probabilities
    return item_popularity, item_probabilities
# -------------------------
# Function to sample negative items based on popularity
def sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=None):
    """Sample N negative items based on popularity, excluding the ground truth."""
    possible_negatives = all_items - set(interacted_item)
    negatives = np.random.choice(
    a=list(possible_negatives),                                    # candidates
    size=min(num_negatives, len(possible_negatives)),              # sample size
    replace=False,                                                 # no duplicates
    p=np.array([item_probabilities.get(item, 0) 
                for item in possible_negatives], dtype=float) / 
      max(1e-12, sum(item_probabilities.get(item, 0) 
                     for item in possible_negatives))              # normalize weights
).tolist()
    
    return negatives
# -------------------------
item_popularity, item_probabilities = load_popularity_data('/Users/baonguyen/IU/thesis/data/counts.csv')
item_probabilities = {item_to_index[key]:value for key,value in item_probabilities.items()}
all_items = [i for i in range(len(unique_item_id)+1)]
all_items=set(all_items)
def evaluate_model(model, val_item_sequences, k=10, max_len=64,  index_to_item=None, device='mps'):
    """
    Evaluate model hit ratio@k on validation data.

    Args:
        model: The trained model.
        val_item_sequences: Dict of user_id -> {'item_id': [...], <feat1>: [...], <feat2>: [...], ...}
        k: Top-k for hit ratio.
        max_len: Sequence length for padding.
        side_feature: List of feature names, e.g. ['Topic', 'category'].
        index_to_item: Dict mapping item indices back to original item ids.
        device: Device to run the model on.

    Returns:
        Hit ratio@k.
    """
    model.eval()
    ground_truths = []
    predictions = []

    with torch.no_grad():
        for user, seq in val_item_sequences.items():
            item_seq = seq['item_id']
            # Prepare input and target
            input_items = item_seq[:-1]
            target_item = item_seq[-1]
            # Pad input sequence (handle all side features)
            mask_seq = {user: {'item_id': input_items}}
            for feat in (side_feature or []):
                mask_seq[user][feat] = seq[feat][:-1]
            padded_seq, _ = padding(
                mask_seq=mask_seq,
                labels={user: []},
                max_len=max_len
            )
            padded_items = padded_seq[user]['item_id']
            # Prepare side feature tensors as a dict
            side_input_dict = {
                feat: torch.tensor([padded_seq[user][feat]], dtype=torch.long).to(device)
                for feat in (side_feature or [])
            }
            item_tensor = torch.tensor([padded_items], dtype=torch.long).to(device)
            key_padding_mask = (item_tensor == 0)
            # Model call
            logits = model(item_tensor, side_input_dict, key_padding_mask=key_padding_mask)[:, min(len(item_seq)-1, max_len-1), :]
            probabilities = torch.softmax(logits, dim=-1)
            # --- Popularity-based Negative Sampling ---
            # Sample N negative items (those not interacted with by the user)
            negatives = sample_negatives_by_popularity(all_items, item_probabilities, num_negatives=100, interacted_item=item_seq)
            candidates = [target_item] + negatives

            # Get probabilities for the candidate items only
            candidate_logits = probabilities[0, candidates]  # Shape: (N+1,)
            
            # Rank candidates by their logits (probabilities)
            ranked = [x for _, x in sorted(zip(candidate_logits.tolist(), candidates), reverse=True)]

            # Store the ground truth and top-k predictions
            ground_truths.append(index_to_item[target_item])
            predictions.append([index_to_item[i] for i in ranked])

            # print(ground_truths)
            # print(predictions)
    return hit_ratio(ground_truths, predictions, k)


In [82]:
import torch
import torch.optim as optim
from tqdm import tqdm
import os
epoch_num = 10
hitrate = 5
def train_model(
    train_users, val_users, fold_num,
    user_item_to_index_sequence, side_feature, unique_item_id,
    embedding_dim=256, max_len=max_len, num_layers=2, num_heads=4, mask_ratio=0.5
):
    train_user_item_to_index_sequence = {user: seq for user, seq in user_item_to_index_sequence.items() if user in train_users}
    mask_seq, labels = mask_sequence(train_user_item_to_index_sequence, mask_ratio=mask_ratio)
    padded_mask_seq, padded_labels = padding(mask_seq, labels, max_len=max_len)

    tensor_item_ids = torch.stack([
        torch.tensor(user_seq['item_id']) for user_seq in padded_mask_seq.values()
    ])
    tensor_side_feats = {
        feat: torch.stack([
            torch.tensor(user_seq[feat]) for user_seq in padded_mask_seq.values()
        ])
        for feat in side_feature
    }

    tensor_labels = torch.stack([
        torch.tensor(seq) for seq in padded_labels.values()
        ])
    train_dataset = torch.utils.data.TensorDataset(
        tensor_item_ids,
        *(tensor_side_feats[feat] for feat in side_feature),
        tensor_labels
    )
    train_dataloader = torch.utils.data.DataLoader(train_dataset,batch_size=64,shuffle=True)
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
    
    num_side_feature_ids = {feat : len(globals()[f'unique_{feat}']) for feat in side_feature} 
    model = NovabertModel(len(unique_item_id)+1,num_side_feature_ids=num_side_feature_ids,embedding_dim=embedding_dim,max_len=max_len,num_layers=num_layers,num_heads=num_heads).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
    best_hr = 0

    for epoch in range(epoch_num):
        model.train()
        epoch_loss = 0
        for batch in tqdm(train_dataloader, desc=f"Fold {fold_num} Epoch {epoch+1}", unit="batch"):
            item_ids = batch[0]
            side_ids = {feat: batch[i+1] for i, feat in enumerate(side_feature)}
            labels = batch[-1]

            # .to(device)
            item_ids = item_ids.to(device)
            side_ids = {feat: tensor.to(device) for feat, tensor in side_ids.items()}
            labels = labels.to(device)

            key_padding_mask = (item_ids == 0)
            optimizer.zero_grad()
            outputs = model(item_ids, side_ids, key_padding_mask=key_padding_mask)
            loss = criterion(outputs.view(-1, len(unique_item_id)+1), labels.view(-1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss {epoch_loss:.4f}")

        model.eval()
        val_item_sequences = {user: seq for user, seq in user_item_to_index_sequence.items() if user in val_users}
        val_hr = evaluate_model(
            model,
            val_item_sequences,
            k=hitrate,
            max_len=max_len,
            index_to_item=index_to_item,
            device=device
        )
        print(f"Fold {fold_num} Epoch {epoch+1}, Validation HR@{hitrate}: {val_hr:.4f}")

        if val_hr > best_hr:
            best_hr = val_hr
            save_path = f"models/models_item_with_novabert_gatingfusor_/fold_{fold_num}"
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/best_model.pth")
    return model, best_hr


In [83]:
from sklearn.model_selection import KFold
import os
import torch

kf = KFold(n_splits=5, shuffle=True, random_state=42)
user_list = list(user_item_sequence.keys())
fold_results = {}

for fold_num, (train_idx, val_idx) in enumerate(kf.split(user_list), 1):
    print(f"\nStarting Fold {fold_num}...")
    train_users = [user_list[i] for i in train_idx]
    val_users = [user_list[i] for i in val_idx]

    # Train model on this fold
    model, val_hr = train_model(
        train_users=train_users,
        val_users=val_users,
        fold_num=fold_num,
        user_item_to_index_sequence=user_item_to_index_sequence,
        side_feature=side_feature,
        unique_item_id=unique_item_id,
    )
    fold_results[fold_num] = val_hr

    save_path = f"results/results_item_with_novabert_gatingfusor_/fold_{fold_num}"
    os.makedirs(save_path, exist_ok=True)
    with open(f"{save_path}/results.txt", "w") as f:
        f.write(f"Validation HR@{hitrate}: {val_hr}\n")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

with open("results/results_item_with_novabert_gatingfusor_/overall_results.txt", "w") as f:
    for fold, hr in fold_results.items():
        f.write(f"Fold {fold}: HR@{hitrate} = {hr}\n")
    mean_hr = sum(fold_results.values()) / len(fold_results)
    f.write(f"\nMean HR@{hitrate} across folds: {mean_hr}\n")

print(f"\nMean HR@{hitrate} across folds: {mean_hr:.4f}")



Starting Fold 1...


Fold 1 Epoch 1: 100%|██████████| 422/422 [00:41<00:00, 10.18batch/s]


Epoch 1, Loss 3458.3490
[Sample 3] GT: 450618, Pred top-5: [450618, 1889597, 1793377, 125465, 166633]
[Sample 17] GT: 1737699, Pred top-5: [174086, 1737699, 467817, 123373, 124553]
[Sample 36] GT: 1949394, Pred top-5: [1949394, 174086, 123373, 166633, 125465]
[Sample 70] GT: 1698166, Pred top-5: [527885, 1956527, 1031440, 1459683, 1698166]
[Sample 76] GT: 450618, Pred top-5: [172027, 126335, 174086, 166633, 450618]
[Sample 106] GT: 126335, Pred top-5: [126335, 136110, 127865, 124553, 123793]
[Sample 117] GT: 125424, Pred top-5: [126335, 174086, 125424, 124553, 137585]
[Sample 141] GT: 127865, Pred top-5: [125465, 137585, 127865, 144051, 123793]
[Sample 152] GT: 136110, Pred top-5: [172027, 174086, 136110, 131117, 127865]
[Sample 153] GT: 708493, Pred top-5: [1460767, 141761, 708493, 1949394, 1576942]
[Sample 154] GT: 131117, Pred top-5: [174086, 136110, 166633, 123373, 131117]
[Sample 179] GT: 1378631, Pred top-5: [1378631, 123373, 125465, 137585, 730008]
[Sample 194] GT: 921642, Pred 

Fold 1 Epoch 2: 100%|██████████| 422/422 [00:41<00:00, 10.24batch/s]


Epoch 2, Loss 3314.9171
[Sample 22] GT: 730008, Pred top-5: [174086, 126335, 123793, 136110, 730008]
[Sample 76] GT: 450618, Pred top-5: [174086, 172027, 127865, 126335, 450618]
[Sample 106] GT: 126335, Pred top-5: [172027, 126335, 125465, 131533, 1213427]
[Sample 141] GT: 127865, Pred top-5: [123793, 127865, 131117, 136860, 123373]
[Sample 152] GT: 136110, Pred top-5: [174086, 123793, 126335, 730008, 136110]
[Sample 154] GT: 131117, Pred top-5: [137585, 136110, 166633, 131117, 132738]
[Sample 160] GT: 132738, Pred top-5: [127865, 148089, 152836, 132738, 123373]
[Sample 177] GT: 127865, Pred top-5: [174086, 126335, 123793, 127865, 730008]
[Sample 179] GT: 1378631, Pred top-5: [174086, 123793, 1076484, 1378631, 145906]
[Sample 186] GT: 148089, Pred top-5: [174086, 123793, 148089, 131117, 1076484]
[Sample 194] GT: 921642, Pred top-5: [172027, 145906, 921642, 126335, 131117]
[Sample 205] GT: 131117, Pred top-5: [136110, 145906, 131117, 132738, 152836]
[Sample 222] GT: 126335, Pred top-5: 

...

Epoch 8, Loss 3038.9708
[Sample 30] GT: 351928, Pred top-5: [1635675, 351928, 1453647, 1188264, 1430716]
[Sample 42] GT: 2530612, Pred top-5: [172027, 1126889, 2530612, 123793, 1982904]
[Sample 51] GT: 1718885, Pred top-5: [999526, 1227811, 1570915, 459535, 1718885]
[Sample 88] GT: 127865, Pred top-5: [127865, 125465, 1076484, 123793, 1954806]
[Sample 102] GT: 1687082, Pred top-5: [127865, 1057664, 1497935, 1687082, 1213427]
[Sample 105] GT: 172027, Pred top-5: [174086, 172027, 123793, 127865, 124553]
[Sample 107] GT: 2956453, Pred top-5: [1750560, 821065, 2956453, 1183835, 281431]
[Sample 115] GT: 730008, Pred top-5: [137585, 172027, 123793, 1076484, 730008]
[Sample 122] GT: 884737, Pred top-5: [1090219, 996851, 1223725, 884737, 1867652]
[Sample 131] GT: 174086, Pred top-5: [174086, 123793, 1076484, 730008, 131533]
[Sample 147] GT: 127865, Pred top-5: [127865, 864981, 131117, 123793, 1325648]
[Sample 149] GT: 1191124, Pred top-5: [1191124, 2557055, 766885, 561215, 1589010]
[Sample 153

Fold 5 Epoch 9: 100%|██████████| 422/422 [00:41<00:00, 10.27batch/s]


Epoch 9, Loss 2993.8800
[Sample 0] GT: 1238932, Pred top-5: [172027, 144051, 1238932, 125424, 125465]
[Sample 3] GT: 166633, Pred top-5: [126335, 197170, 730008, 166633, 132738]
[Sample 30] GT: 351928, Pred top-5: [561215, 351928, 394079, 2940176, 1419447]
[Sample 49] GT: 484069, Pred top-5: [1225430, 1744232, 1698815, 1340234, 484069]
[Sample 51] GT: 1718885, Pred top-5: [938882, 797218, 1718885, 2216225, 1861964]
[Sample 56] GT: 166633, Pred top-5: [174086, 166633, 136110, 145906, 123793]
[Sample 71] GT: 125465, Pred top-5: [123793, 125465, 746366, 1308013, 172027]
[Sample 88] GT: 127865, Pred top-5: [127865, 166633, 131117, 172027, 125465]
[Sample 91] GT: 2231364, Pred top-5: [1661651, 2231364, 1435687, 364092, 234276]
[Sample 101] GT: 1445609, Pred top-5: [1687910, 773361, 631251, 1445609, 1566870]
[Sample 105] GT: 172027, Pred top-5: [172027, 126335, 123793, 131117, 124553]
[Sample 116] GT: 152836, Pred top-5: [174086, 123793, 145906, 152836, 130259]
[Sample 131] GT: 174086, Pred 

Fold 5 Epoch 10: 100%|██████████| 422/422 [00:41<00:00, 10.29batch/s]


Epoch 10, Loss 2945.2023
[Sample 22] GT: 1698166, Pred top-5: [1698166, 1463543, 136860, 1515649, 1695878]
[Sample 30] GT: 351928, Pred top-5: [1129763, 1893305, 351928, 1741645, 2778191]
[Sample 51] GT: 1718885, Pred top-5: [2686855, 842449, 2864831, 827522, 1718885]
[Sample 71] GT: 125465, Pred top-5: [921642, 172027, 123793, 1992625, 125465]
[Sample 88] GT: 127865, Pred top-5: [127865, 131533, 126335, 174086, 124553]
[Sample 104] GT: 1076484, Pred top-5: [131533, 1126889, 1076484, 172027, 128959]
[Sample 105] GT: 172027, Pred top-5: [136110, 174086, 172027, 131533, 963476]
[Sample 107] GT: 2956453, Pred top-5: [2003299, 2660249, 1783282, 2956453, 2117415]
[Sample 131] GT: 174086, Pred top-5: [174086, 136110, 136860, 137585, 123793]
[Sample 132] GT: 580060, Pred top-5: [127865, 451969, 580060, 2834590, 1505204]
[Sample 147] GT: 127865, Pred top-5: [127865, 1366699, 503972, 131533, 123793]
[Sample 149] GT: 1191124, Pred top-5: [1191124, 1008472, 1492185, 2771965, 1528337]
[Sample 171]